# Capstone: Castalia Scholar — Full System Orchestration

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/agents/37_project_full_system.ipynb)

## Module 7 — Capstone Project (Notebook 6 of 6)

This is the **final notebook** in the Castalia Agents course. Everything we've built across 36 notebooks comes together here: prompt engineering, tool use, ReAct loops, memory, RAG, planning, reflection, multi-agent coordination — all orchestrated into a single, working research system.

**Castalia Scholar** is a multi-agent research assistant. Give it a question, and it:
1. **Plans** the research by decomposing the question into sub-questions
2. **Retrieves** relevant documents from a knowledge corpus
3. **Analyzes** the findings — synthesis, contradiction detection, gap identification
4. **Writes** a structured research report with citations
5. **Reviews** the report for quality — and sends it back for revision if needed
6. **Delivers** a polished final report

### Learning Objectives

By the end of this notebook you will:
1. **Build and run a complete multi-agent system** end-to-end
2. **Orchestrate 5 specialized agents** with state management and error handling
3. **Implement a revision feedback loop** where the reviewer drives quality improvement
4. **Evaluate system performance** across multiple research questions
5. **Compare the full system against simpler approaches** to see the value of multi-agent orchestration

---

> **Prerequisites**: Notebooks 01–36 (entire course).
> **Runtime**: Google Colab with T4 GPU.
> **Time**: ~60–90 minutes (the full system runs multiple agents per question).

## Part 0: Environment Setup

This notebook needs both the LLM (for all agents) and the embedding model (for retrieval). We use FAISS for vector search over our document corpus.

In [ ]:
# --- Google Colab Setup ---
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch sentence-transformers faiss-cpu numpy

import torch
import time
import json
import re
import math
import numpy as np
import faiss
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any, Callable, Tuple, Union
from collections import defaultdict
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer

# Mount Google Drive for model caching
drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, device_map="auto", quantization_config=quantization_config,
    cache_dir=CACHE_DIR, torch_dtype="auto",
)

# Load embedding model for memory/retrieval
embed_model = SentenceTransformer("BAAI/bge-base-en-v1.5", cache_folder=CACHE_DIR)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs, max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample, top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

def embed_texts(texts):
    """Embed a list of texts using BGE model. Returns numpy array."""
    return embed_model.encode(texts, normalize_embeddings=True)

print(f"✓ LLM loaded: {MODEL_NAME}")
print(f"  GPU memory: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")
print(f"✓ Embedding model loaded: BAAI/bge-base-en-v1.5 (768-dim)")

## Part 1: System Architecture Review

Here is the complete Castalia Scholar architecture — the system we're about to build:

```
                         ┌─────────────────────┐
                         │    USER QUESTION     │
                         └──────────┬──────────┘
                                    │
                                    ▼
                    ┌───────────────────────────────┐
                    │       ORCHESTRATOR AGENT       │
                    │                                │
                    │  • Manages workflow stages      │
                    │  • Tracks state & timing        │
                    │  • Handles errors & retries     │
                    │  • Enforces revision limits     │
                    └───────────────┬───────────────┘
                                    │
              ┌─────────────────────┼─────────────────────┐
              │                     │                     │
              ▼                     ▼                     ▼
     ┌─────────────┐     ┌─────────────┐     ┌─────────────┐
     │   PLANNER   │     │  RETRIEVER  │     │  ANALYZER   │
     │             │     │             │     │             │
     │ Decompose   │────►│ FAISS search│────►│ Synthesize  │
     │ into sub-Qs │     │ + reranking │     │ + detect    │
     └─────────────┘     └─────────────┘     │ conflicts   │
                                              └──────┬──────┘
                                                     │
                                                     ▼
                                              ┌─────────────┐
                                              │   WRITER    │
                                              │             │
                                              │ Report with │
                                              │ citations   │
                                              └──────┬──────┘
                                                     │
                                                     ▼
                                              ┌─────────────┐
                                         ┌───►│  REVIEWER   │
                                         │    │             │
                                         │    │ 5-dimension │
                                         │    │ evaluation  │
                                         │    └──────┬──────┘
                                         │           │
                                         │     ┌─────┴─────┐
                                         │     │           │
                                         │   REVISE      PASS
                                         │     │           │
                                         │     ▼           ▼
                                         └── Writer    FINAL REPORT
                                         (max 2 rounds)
```

### Data Flow

| Stage | Input | Output | Agent |
|-------|-------|--------|-------|
| 1. Plan | User question | List of sub-questions | PlannerAgent |
| 2. Retrieve | Sub-questions | Relevant passages + scores | RetrieverAgent |
| 3. Analyze | Passages + question | Synthesis + contradictions + gaps | AnalyzerAgent |
| 4. Write | Analysis + sources + question | Structured report with citations | WriterAgent |
| 5. Review | Report + sources + question | Scores + feedback + decision | ReviewerAgent |
| 5b. Revise | Report + feedback | Improved report | WriterAgent (again) |

## Part 2: Building the Knowledge Corpus

We create an inline corpus of 20+ passages on AI/ML topics. These serve as the "documents" our Retrieval Agent searches. Each passage contains factual content suitable for research questions about AI.

In [ ]:
# ── AI/ML Knowledge Corpus ──
# Each passage is a factual text on a specific AI/ML topic.

CORPUS = [
    {
        "id": "neural_nets_basics",
        "title": "Neural Network Fundamentals",
        "text": (
            "Neural networks are computational models inspired by biological neurons. "
            "A neural network consists of layers of interconnected nodes (neurons), where each "
            "connection has a learnable weight. The basic architecture includes an input layer, "
            "one or more hidden layers, and an output layer. During forward propagation, input "
            "data flows through the network, with each neuron computing a weighted sum of its "
            "inputs followed by a non-linear activation function such as ReLU, sigmoid, or tanh. "
            "Training uses backpropagation to compute gradients of a loss function with respect "
            "to each weight, then updates weights using gradient descent. The universal "
            "approximation theorem proves that a network with a single hidden layer can "
            "approximate any continuous function, though deeper networks are more efficient "
            "in practice. Modern neural networks often have dozens or hundreds of layers, "
            "enabled by techniques like batch normalization, residual connections, and "
            "careful initialization strategies like Xavier or He initialization."
        ),
    },
    {
        "id": "transformers_architecture",
        "title": "The Transformer Architecture",
        "text": (
            "The Transformer, introduced by Vaswani et al. in 2017 ('Attention Is All You Need'), "
            "replaced recurrent architectures with a purely attention-based mechanism. The key "
            "innovation is self-attention (scaled dot-product attention), which computes attention "
            "weights as softmax(QK^T / sqrt(d_k))V, where Q, K, V are query, key, and value "
            "matrices derived from the input. Multi-head attention runs multiple attention "
            "operations in parallel, each with different learned projections, allowing the model "
            "to attend to information from different representation subspaces. The original "
            "Transformer uses 8 attention heads, a model dimension of 512, and 6 encoder/decoder "
            "layers. Positional encoding is added to input embeddings since attention is "
            "permutation-invariant. The encoder processes the full input sequence, while the "
            "decoder generates output autoregressively with masked attention to prevent looking "
            "ahead. Layer normalization and residual connections stabilize training of deep "
            "Transformer models."
        ),
    },
    {
        "id": "attention_mechanism",
        "title": "Attention Mechanisms in Deep Learning",
        "text": (
            "Attention mechanisms allow neural networks to focus on relevant parts of the input "
            "when producing each part of the output. Bahdanau et al. (2014) introduced additive "
            "attention for machine translation, computing alignment scores between encoder hidden "
            "states and the current decoder state. Luong et al. (2015) proposed multiplicative "
            "(dot-product) attention as a simpler alternative. Self-attention, used in "
            "Transformers, computes attention within a single sequence, enabling each position "
            "to attend to all other positions. Cross-attention computes attention between two "
            "different sequences, used in encoder-decoder models. Multi-head attention splits "
            "the attention computation into multiple parallel heads, each operating on a different "
            "linear projection of the input. Flash Attention (Dao et al., 2022) optimizes the "
            "attention computation using tiling and kernel fusion for better GPU memory efficiency, "
            "enabling longer context windows. Sparse attention patterns like those in Longformer "
            "and BigBird reduce the quadratic complexity of standard attention."
        ),
    },
    {
        "id": "reinforcement_learning_basics",
        "title": "Reinforcement Learning Fundamentals",
        "text": (
            "Reinforcement learning (RL) trains agents to make sequences of decisions by "
            "maximizing cumulative reward. The agent interacts with an environment modeled as "
            "a Markov Decision Process (MDP) with states, actions, transition probabilities, "
            "and rewards. Value-based methods like Q-learning estimate the expected return for "
            "each state-action pair. The Q-function satisfies the Bellman equation: "
            "Q(s,a) = r + gamma * max_a' Q(s',a'). Deep Q-Networks (DQN) by Mnih et al. (2015) "
            "use neural networks to approximate Q-values, with experience replay and target "
            "networks for stability. Policy-based methods like REINFORCE directly optimize "
            "the policy using the policy gradient theorem. Actor-critic methods combine both: "
            "the actor learns the policy while the critic estimates the value function. "
            "Proximal Policy Optimization (PPO) by Schulman et al. (2017) constrains policy "
            "updates to prevent catastrophically large changes, making training more stable."
        ),
    },
    {
        "id": "rl_advanced",
        "title": "Advanced Reinforcement Learning",
        "text": (
            "Model-based RL learns a dynamics model of the environment and uses it for planning. "
            "MuZero (Schrittwieser et al., 2020) learns a latent dynamics model and achieves "
            "superhuman performance on Atari, Go, and chess without knowing the rules. "
            "Multi-agent RL extends to settings with multiple interacting agents, introducing "
            "challenges of non-stationarity and credit assignment. Offline RL (batch RL) learns "
            "from fixed datasets without environment interaction, useful when real-world "
            "interaction is expensive or dangerous. Conservative Q-Learning (CQL) penalizes "
            "overestimation of out-of-distribution actions. Hierarchical RL decomposes complex "
            "tasks into sub-goals, with higher-level policies selecting sub-goals and lower-level "
            "policies achieving them. Reward shaping and intrinsic motivation (curiosity-driven "
            "exploration) help agents explore sparse-reward environments. Inverse RL infers "
            "reward functions from expert demonstrations, enabling learning from observation."
        ),
    },
    {
        "id": "nlp_history",
        "title": "History of Natural Language Processing",
        "text": (
            "NLP has evolved through several paradigms. Rule-based systems (1950s-1980s) used "
            "hand-crafted grammars and dictionaries. Statistical NLP (1990s-2010s) introduced "
            "probabilistic models: n-gram language models, Hidden Markov Models for POS tagging, "
            "and statistical machine translation. Word embeddings like Word2Vec (Mikolov et al., "
            "2013) and GloVe (Pennington et al., 2014) represented words as dense vectors "
            "capturing semantic relationships. The neural NLP era began with recurrent neural "
            "networks (RNNs) and LSTMs for sequence modeling. ELMo (Peters et al., 2018) "
            "introduced contextualized word representations. BERT (Devlin et al., 2019) "
            "revolutionized NLP with bidirectional pre-training on masked language modeling, "
            "achieving state-of-the-art on 11 NLP benchmarks. GPT (Radford et al., 2018) "
            "demonstrated that autoregressive pre-training followed by fine-tuning could match "
            "supervised approaches. The scaling era brought GPT-3 (175B parameters), PaLM "
            "(540B), and instruction-tuned models like ChatGPT."
        ),
    },
    {
        "id": "computer_vision_cnns",
        "title": "Convolutional Neural Networks for Computer Vision",
        "text": (
            "Convolutional Neural Networks (CNNs) are the foundation of modern computer vision. "
            "LeNet-5 (LeCun et al., 1998) pioneered the architecture for digit recognition. "
            "AlexNet (Krizhevsky et al., 2012) won ImageNet with a deep CNN using ReLU "
            "activations and dropout, igniting the deep learning revolution. VGGNet (Simonyan "
            "& Zisserman, 2014) showed that deeper networks with small 3x3 filters outperform "
            "shallower ones. GoogLeNet/Inception (Szegedy et al., 2014) introduced inception "
            "modules with parallel convolutions at different scales. ResNet (He et al., 2015) "
            "solved the vanishing gradient problem in very deep networks using skip connections, "
            "enabling training of 152+ layer networks. DenseNet connects each layer to every "
            "other layer in a feed-forward fashion. EfficientNet (Tan & Le, 2019) systematically "
            "scales depth, width, and resolution using compound scaling, achieving better "
            "accuracy with fewer parameters."
        ),
    },
    {
        "id": "vision_transformers",
        "title": "Vision Transformers",
        "text": (
            "Vision Transformers (ViT) by Dosovitskiy et al. (2020) applied the Transformer "
            "architecture directly to image classification. Images are split into fixed-size "
            "patches (e.g., 16x16), which are linearly embedded and treated as a sequence of "
            "tokens. A learnable [CLS] token aggregates information for classification. ViT "
            "requires large-scale pre-training (ImageNet-21k or JFT-300M) to outperform CNNs, "
            "but when trained with sufficient data, it achieves state-of-the-art results. "
            "DeiT (Touvron et al., 2021) introduced training strategies that allow ViTs to "
            "be trained effectively on ImageNet-1k alone using knowledge distillation. Swin "
            "Transformer (Liu et al., 2021) introduces shifted window attention for linear "
            "complexity and hierarchical feature maps, making it suitable for dense prediction "
            "tasks like object detection and segmentation. BEiT applies BERT-style masked "
            "image modeling for self-supervised pre-training of vision transformers."
        ),
    },
    {
        "id": "gans",
        "title": "Generative Adversarial Networks",
        "text": (
            "Generative Adversarial Networks (GANs), introduced by Goodfellow et al. (2014), "
            "train two networks in competition: a generator G that produces synthetic data and "
            "a discriminator D that distinguishes real from fake. The training objective is a "
            "minimax game: min_G max_D E[log D(x)] + E[log(1-D(G(z)))]. Training challenges "
            "include mode collapse (generator produces limited variety) and training instability. "
            "Wasserstein GAN (Arjovsky et al., 2017) uses Earth Mover distance for more stable "
            "training. Progressive GAN (Karras et al., 2018) trains at increasing resolutions. "
            "StyleGAN (Karras et al., 2019) introduces a style-based generator with adaptive "
            "instance normalization, producing photorealistic face images at 1024x1024 "
            "resolution. Conditional GANs (cGAN) generate images conditioned on class labels "
            "or input images, enabling tasks like image-to-image translation (Pix2Pix) and "
            "style transfer (CycleGAN)."
        ),
    },
    {
        "id": "diffusion_models",
        "title": "Diffusion Models for Generation",
        "text": (
            "Diffusion models generate data by learning to reverse a gradual noising process. "
            "Denoising Diffusion Probabilistic Models (DDPM) by Ho et al. (2020) add Gaussian "
            "noise to data over T steps, then train a neural network to predict and remove the "
            "noise at each step. The reverse process generates samples by iteratively denoising "
            "from pure noise. Score-based models (Song & Ermon, 2019) learn the gradient of the "
            "data distribution (score function) and generate samples via Langevin dynamics. "
            "Latent Diffusion Models (Rombach et al., 2022) operate in a compressed latent "
            "space rather than pixel space, dramatically reducing computational cost. This "
            "approach powers Stable Diffusion. Classifier-free guidance (Ho & Salimans, 2022) "
            "improves sample quality by interpolating between conditional and unconditional "
            "predictions. DALL-E 2 and Imagen use diffusion models with text conditioning "
            "for text-to-image generation, achieving remarkable photorealistic results."
        ),
    },
    {
        "id": "llm_pretraining",
        "title": "Pre-training Large Language Models",
        "text": (
            "Large Language Models are pre-trained on massive text corpora using self-supervised "
            "objectives. Causal language modeling (next-token prediction) is used by GPT-series "
            "models: given tokens t_1...t_n, predict t_{n+1}. Masked language modeling (MLM), "
            "used by BERT, randomly masks 15%% of tokens and predicts them from context. "
            "Pre-training data includes web crawls (Common Crawl, C4), books (BookCorpus, "
            "Gutenberg), code (GitHub), and curated datasets (The Pile, RedPajama). Data "
            "quality matters enormously: deduplication removes repeated content, quality "
            "filtering removes low-quality text, and domain balancing ensures diverse coverage. "
            "Scaling laws (Kaplan et al., 2020) show that loss decreases as a power law with "
            "model size, dataset size, and compute. Chinchilla (Hoffmann et al., 2022) proved "
            "that many models were undertrained — a 70B model trained on more data outperforms "
            "a 280B model trained on less data."
        ),
    },
    {
        "id": "llm_finetuning",
        "title": "Fine-tuning and Alignment of LLMs",
        "text": (
            "After pre-training, LLMs undergo fine-tuning to follow instructions and align with "
            "human preferences. Supervised Fine-Tuning (SFT) trains on curated instruction-response "
            "pairs. RLHF (Reinforcement Learning from Human Feedback) trains a reward model on "
            "human preference data, then optimizes the LLM policy using PPO to maximize the "
            "learned reward while staying close to the SFT model via a KL penalty. DPO (Direct "
            "Preference Optimization) by Rafailov et al. (2023) simplifies RLHF by directly "
            "optimizing the policy from preferences without a separate reward model. "
            "Constitutional AI (Anthropic) uses a set of principles to generate self-critiques "
            "and revisions, reducing the need for human feedback. LoRA (Hu et al., 2021) enables "
            "parameter-efficient fine-tuning by training low-rank adapter matrices instead of "
            "all model weights, reducing memory requirements by 10-100x while maintaining "
            "performance close to full fine-tuning."
        ),
    },
    {
        "id": "distributed_training",
        "title": "Distributed Training of Large Models",
        "text": (
            "Training models with billions of parameters requires distributed computing across "
            "multiple GPUs and nodes. Data parallelism replicates the model on each GPU and "
            "splits the batch, averaging gradients after each step. ZeRO (Rajbhandari et al., "
            "2020) partitions optimizer states, gradients, and parameters across GPUs to reduce "
            "memory redundancy. Tensor parallelism (Megatron-LM) splits individual layers "
            "across GPUs, enabling models too large for a single GPU. Pipeline parallelism "
            "splits the model by layers across GPUs, processing micro-batches in a pipelined "
            "fashion. Mixed precision training uses FP16 or BF16 for computation with FP32 "
            "master weights, halving memory and doubling throughput. Gradient checkpointing "
            "trades compute for memory by recomputing activations during the backward pass "
            "instead of storing them. Training GPT-3 (175B) required 1024 A100 GPUs running "
            "for approximately 34 days."
        ),
    },
    {
        "id": "rag_retrieval",
        "title": "Retrieval-Augmented Generation",
        "text": (
            "Retrieval-Augmented Generation (RAG) enhances LLM outputs by incorporating "
            "retrieved external knowledge. The basic RAG pipeline: (1) encode the query using "
            "an embedding model, (2) search a vector database for similar documents, (3) include "
            "retrieved documents in the LLM prompt as context. Dense retrieval models like DPR "
            "(Karpukhin et al., 2020) and BGE (Xiao et al., 2023) produce embeddings optimized "
            "for semantic similarity search. FAISS (Johnson et al., 2017) provides efficient "
            "approximate nearest-neighbor search for large-scale vector databases. Advanced RAG "
            "techniques include query decomposition (breaking complex queries into sub-queries), "
            "hybrid search (combining dense and sparse retrieval), and reranking (using a "
            "cross-encoder to re-score retrieved candidates). Self-RAG (Asai et al., 2023) "
            "trains the LLM to decide when to retrieve and to critique its own use of "
            "retrieved information."
        ),
    },
    {
        "id": "prompt_engineering",
        "title": "Prompt Engineering Techniques",
        "text": (
            "Prompt engineering designs input prompts to elicit desired LLM behavior without "
            "modifying model weights. Zero-shot prompting provides only the task description. "
            "Few-shot prompting includes examples of input-output pairs in the prompt. "
            "Chain-of-thought (CoT) prompting (Wei et al., 2022) instructs the model to reason "
            "step-by-step before answering, dramatically improving performance on reasoning "
            "tasks. Self-consistency (Wang et al., 2023) samples multiple reasoning paths and "
            "takes the majority vote. Tree-of-thought prompting explores multiple reasoning "
            "branches explicitly. ReAct (Yao et al., 2023) interleaves reasoning and action "
            "steps, enabling the model to use tools and ground its reasoning. System prompts "
            "define the model's persona and constraints. Role prompting assigns specific "
            "expertise to the model. Output formatting instructions (e.g., requesting JSON) "
            "improve structured output reliability."
        ),
    },
    {
        "id": "multi_agent_systems",
        "title": "Multi-Agent AI Systems",
        "text": (
            "Multi-agent systems use multiple specialized AI agents that communicate and "
            "collaborate to solve complex tasks. Each agent typically has a defined role, "
            "specialized tools, and a system prompt. Orchestration patterns include: (1) "
            "sequential pipelines where agents process in order, (2) hierarchical systems with "
            "a supervisor agent delegating to workers, (3) debate systems where agents argue "
            "different positions. AutoGen (Wu et al., 2023) provides a framework for multi-agent "
            "conversations. CrewAI organizes agents into crews with defined roles and workflows. "
            "LangGraph enables building agent workflows as state machines with conditional "
            "branching. Key challenges include managing shared state, handling failures gracefully, "
            "preventing error propagation between agents, and controlling token usage. "
            "Multi-agent systems excel at complex research, code generation with review, and "
            "tasks requiring multiple perspectives or expertise areas."
        ),
    },
    {
        "id": "transfer_learning",
        "title": "Transfer Learning in Deep Learning",
        "text": (
            "Transfer learning leverages knowledge from one task to improve performance on "
            "another. In computer vision, models pre-trained on ImageNet are fine-tuned for "
            "specific tasks by replacing the final classification layer and training with a "
            "lower learning rate. Feature extraction uses the pre-trained model as a fixed "
            "feature extractor. In NLP, pre-trained language models (BERT, GPT) are fine-tuned "
            "on downstream tasks like sentiment analysis, named entity recognition, and question "
            "answering. Domain adaptation addresses distribution shift between source and target "
            "domains. Few-shot learning aims to learn from very few examples, often using "
            "meta-learning approaches like MAML (Finn et al., 2017) that learn initialization "
            "parameters that can quickly adapt to new tasks. Instruction tuning is a form of "
            "transfer learning where a pre-trained LLM is further trained on diverse "
            "instruction-following datasets to generalize to unseen tasks."
        ),
    },
    {
        "id": "model_evaluation",
        "title": "Evaluating AI Models",
        "text": (
            "Model evaluation measures how well a model performs on its intended task. "
            "Classification metrics include accuracy, precision, recall, F1-score, and "
            "area under the ROC curve (AUC-ROC). For language generation, BLEU measures "
            "n-gram overlap with reference translations. ROUGE evaluates summarization quality. "
            "Perplexity measures how well a language model predicts text (lower is better). "
            "Human evaluation remains the gold standard for generation quality but is expensive "
            "and slow. LLM-as-judge uses a capable LLM to evaluate other models' outputs, "
            "correlating well with human judgments for many tasks. Benchmark suites like MMLU "
            "(Massive Multitask Language Understanding), HellaSwag (commonsense reasoning), "
            "HumanEval (code generation), and MATH (mathematical reasoning) provide standardized "
            "evaluation. Contamination and benchmark saturation are growing concerns as models "
            "may have seen test data during pre-training."
        ),
    },
    {
        "id": "ai_safety",
        "title": "AI Safety and Alignment",
        "text": (
            "AI safety research ensures AI systems behave as intended and remain beneficial. "
            "Alignment aims to make AI systems pursue goals that match human values and "
            "intentions. RLHF is currently the primary technique for aligning language models. "
            "Red teaming systematically probes models for harmful outputs, biases, and "
            "vulnerabilities. Constitutional AI defines explicit principles the model should "
            "follow. Scalable oversight addresses the challenge of supervising AI systems "
            "that may exceed human capability in specific domains. Interpretability research "
            "aims to understand what models learn and how they make decisions, using techniques "
            "like attention visualization, probing classifiers, and mechanistic interpretability. "
            "Adversarial robustness ensures models handle adversarial inputs gracefully. The "
            "alignment tax refers to the performance cost of making models safer, which "
            "researchers aim to minimize."
        ),
    },
    {
        "id": "embeddings_representations",
        "title": "Embeddings and Learned Representations",
        "text": (
            "Embeddings map discrete objects (words, images, users) to continuous vector spaces "
            "where similar items are close together. Word2Vec (Mikolov et al., 2013) learns "
            "word embeddings using skip-gram or CBOW objectives. Sentence embeddings (from "
            "models like Sentence-BERT, BGE, E5) represent entire sentences as fixed-length "
            "vectors useful for semantic search and clustering. Image embeddings from CLIP "
            "(Radford et al., 2021) align visual and textual representations in a shared space, "
            "enabling zero-shot image classification and text-to-image retrieval. Contrastive "
            "learning trains embeddings by pulling similar pairs together and pushing dissimilar "
            "pairs apart. Matryoshka Representation Learning allows embeddings to be truncated "
            "to smaller dimensions while preserving most of their quality. Embedding models "
            "are evaluated on benchmarks like MTEB (Massive Text Embedding Benchmark) which "
            "tests retrieval, classification, clustering, and semantic similarity tasks."
        ),
    },
]

print(f"✓ Corpus built: {len(CORPUS)} passages")
for i, doc in enumerate(CORPUS):
    print(f"  [{i:2d}] {doc['title']:<45s} ({len(doc['text'])} chars)")

### 2.1 — Index the Corpus with FAISS + BGE Embeddings

We embed all passages and build a FAISS index for fast semantic search.

In [ ]:
# Embed all corpus passages
corpus_texts = [doc["text"] for doc in CORPUS]
corpus_embeddings = embed_texts(corpus_texts)

# Build FAISS index
embedding_dim = corpus_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(embedding_dim)  # inner product (cosine on normalized vecs)
faiss_index.add(corpus_embeddings.astype(np.float32))

print(f"✓ FAISS index built")
print(f"  Documents: {faiss_index.ntotal}")
print(f"  Embedding dim: {embedding_dim}")
print(f"  Index type: Flat Inner Product (exact cosine search)")

# Quick test: search for "transformer"
test_query = embed_texts(["How do transformers work?"])
scores, indices = faiss_index.search(test_query.astype(np.float32), k=3)
print(f"\nTest search: 'How do transformers work?'")
for i, (score, idx) in enumerate(zip(scores[0], indices[0])):
    print(f"  {i+1}. [{score:.3f}] {CORPUS[idx]['title']}")

## Part 3: Rebuilding Key Components

We rebuild simplified versions of each agent so this notebook runs standalone. Each agent follows the same pattern: take input → construct a prompt → call the LLM → parse the output.

In [ ]:
class PlannerAgent:
    """Decomposes a research question into focused sub-questions."""

    def plan(self, question: str, verbose: bool = True) -> List[str]:
        """Break a research question into 3-5 sub-questions."""
        messages = [
            {"role": "system", "content":
                "You are a research planning agent. Given a research question, decompose it "
                "into 3-5 focused sub-questions that together would fully answer the original "
                "question. Each sub-question should target a specific aspect.\n\n"
                "Respond with JSON: {\"sub_questions\": [\"q1\", \"q2\", ...]}"},
            {"role": "user", "content": f"Research question: {question}"},
        ]
        raw = generate(messages, max_new_tokens=300, temperature=0.5, do_sample=True)

        sub_questions = []
        try:
            match = re.search(r'\{[\s\S]*\}', raw)
            if match:
                data = json.loads(match.group())
                sub_questions = data.get("sub_questions", [])
        except (json.JSONDecodeError, ValueError):
            # Fallback: extract lines that look like questions
            sub_questions = [line.strip() for line in raw.split("\n")
                            if line.strip().endswith("?")]

        if not sub_questions:
            sub_questions = [question]  # fallback to original question

        if verbose:
            print(f"[Planner] Decomposed into {len(sub_questions)} sub-questions:")
            for i, sq in enumerate(sub_questions, 1):
                print(f"  {i}. {sq}")

        return sub_questions

planner = PlannerAgent()
print("✓ PlannerAgent ready")

In [ ]:
class RetrieverAgent:
    """Searches the corpus for passages relevant to each sub-question."""

    def __init__(self, index, corpus, embed_fn, top_k: int = 3):
        self.index = index
        self.corpus = corpus
        self.embed_fn = embed_fn
        self.top_k = top_k

    def retrieve(self, sub_questions: List[str],
                 verbose: bool = True) -> List[dict]:
        """Retrieve relevant passages for each sub-question.

        Returns a deduplicated list of {doc_id, title, text, score, matched_query}.
        """
        all_results = {}

        for sq in sub_questions:
            query_emb = self.embed_fn([sq]).astype(np.float32)
            scores, indices = self.index.search(query_emb, k=self.top_k)

            for score, idx in zip(scores[0], indices[0]):
                doc = self.corpus[idx]
                doc_id = doc["id"]
                if doc_id not in all_results or score > all_results[doc_id]["score"]:
                    all_results[doc_id] = {
                        "doc_id": doc_id,
                        "title": doc["title"],
                        "text": doc["text"],
                        "score": float(score),
                        "matched_query": sq,
                    }

        # Sort by score and return
        results = sorted(all_results.values(), key=lambda x: x["score"], reverse=True)

        if verbose:
            print(f"[Retriever] Found {len(results)} unique passages from {len(sub_questions)} queries:")
            for r in results:
                print(f"  [{r['score']:.3f}] {r['title']}")

        return results

retriever = RetrieverAgent(faiss_index, CORPUS, embed_texts, top_k=3)
print("✓ RetrieverAgent ready")

In [ ]:
class AnalyzerAgent:
    """Synthesizes retrieved passages into structured analysis."""

    def analyze(self, question: str, passages: List[dict],
                verbose: bool = True) -> dict:
        """Analyze retrieved passages: synthesize, detect contradictions, find gaps."""
        passages_text = ""
        for i, p in enumerate(passages):
            passages_text += f"\n[Source {i+1}: {p['title']}]\n{p['text']}\n"

        messages = [
            {"role": "system", "content":
                "You are a research analysis agent. Given a question and source passages, produce "
                "a structured analysis. Respond with JSON:\n"
                "{\"key_findings\": [\"finding1\", ...], "
                "\"themes\": [{\"name\": \"...\", \"sources\": [1,2], \"summary\": \"...\"}], "
                "\"contradictions\": [\"...\"] or [], "
                "\"gaps\": [\"...\"] or [], "
                "\"synthesis\": \"overall synthesis paragraph\"}"},
            {"role": "user", "content":
                f"Research question: {question}\n\nSource passages:{passages_text}"},
        ]
        raw = generate(messages, max_new_tokens=700, temperature=0.5, do_sample=True)

        try:
            match = re.search(r'\{[\s\S]*\}', raw)
            if match:
                analysis = json.loads(match.group())
            else:
                analysis = {"synthesis": raw, "key_findings": [], "themes": [],
                            "contradictions": [], "gaps": []}
        except (json.JSONDecodeError, ValueError):
            analysis = {"synthesis": raw[:500], "key_findings": [], "themes": [],
                        "contradictions": [], "gaps": []}

        if verbose:
            print(f"[Analyzer] Analysis complete:")
            print(f"  Key findings: {len(analysis.get('key_findings', []))}")
            print(f"  Themes: {len(analysis.get('themes', []))}")
            print(f"  Contradictions: {len(analysis.get('contradictions', []))}")
            print(f"  Gaps: {len(analysis.get('gaps', []))}")

        return analysis

analyzer = AnalyzerAgent()
print("✓ AnalyzerAgent ready")

In [ ]:
class WriterAgent:
    """Produces a structured research report with citations."""

    def write(self, question: str, analysis: dict,
              passages: List[dict], verbose: bool = True) -> str:
        """Write a research report based on the analysis and source passages."""
        # Build source reference list
        source_refs = "\n".join(
            f"[Source {i+1}] {p['title']}" for i, p in enumerate(passages)
        )
        analysis_text = json.dumps(analysis, indent=2)[:1500]

        messages = [
            {"role": "system", "content":
                "You are a research report writer. Write a clear, well-structured report "
                "that answers the research question using the analysis and sources provided.\n\n"
                "Requirements:\n"
                "- Include an introduction, 2-3 body sections organized by theme, and a conclusion\n"
                "- Cite sources as [Source N] when using information from them\n"
                "- Be factually accurate — only include claims supported by the sources\n"
                "- Write in an academic but accessible style"},
            {"role": "user", "content":
                f"Research question: {question}\n\n"
                f"Analysis:\n{analysis_text}\n\n"
                f"Available sources:\n{source_refs}\n\n"
                f"Write the complete report:"},
        ]
        report = generate(messages, max_new_tokens=800, temperature=0.7)

        if verbose:
            print(f"[Writer] Report generated: {len(report)} chars")
            sections = re.findall(r'^#+\s+(.+)', report, re.MULTILINE)
            if sections:
                print(f"  Sections: {', '.join(sections[:5])}")

        return report

    def revise(self, report: str, feedback: str, question: str,
               passages: List[dict], verbose: bool = True) -> str:
        """Revise a report based on reviewer feedback."""
        source_refs = "\n".join(
            f"[Source {i+1}] {p['title']}" for i, p in enumerate(passages)
        )
        messages = [
            {"role": "system", "content":
                "You are revising a research report based on reviewer feedback. "
                "Fix all identified issues while keeping what was good. "
                "Produce the COMPLETE revised report — not just the changes."},
            {"role": "user", "content":
                f"QUESTION: {question}\n\n"
                f"CURRENT REPORT:\n{report}\n\n"
                f"REVIEWER FEEDBACK:\n{feedback}\n\n"
                f"SOURCES:\n{source_refs}\n\n"
                f"Produce the complete revised report:"},
        ]
        revised = generate(messages, max_new_tokens=800, temperature=0.7)
        if verbose:
            print(f"[Writer] Revised report: {len(revised)} chars")
        return revised

writer = WriterAgent()
print("✓ WriterAgent ready")

In [ ]:
@dataclass
class ReviewScore:
    """Score for one review dimension."""
    dimension: str
    score: int
    issues: List[str]
    suggestions: List[str]

class ReviewerAgent:
    """Evaluates report quality across multiple dimensions."""

    DIMENSIONS = ["accuracy", "completeness", "clarity", "citation_quality", "coherence"]

    def review(self, report: str, sources: List[dict],
               question: str, verbose: bool = True) -> dict:
        """Review a report across all quality dimensions.

        Returns dict with scores, decision, and feedback.
        """
        src_text = "\n".join(
            f"[Source {i+1}: {s['title']}] {s['text'][:200]}..."
            for i, s in enumerate(sources)
        )

        messages = [
            {"role": "system", "content":
                "You are a research report reviewer. Evaluate the report across these "
                "dimensions: accuracy, completeness, clarity, citation_quality, coherence.\n\n"
                "Score each dimension 1-10. Identify specific issues and suggestions.\n"
                "Decide PASS (all dimensions >= 6 and average >= 7) or REVISE.\n\n"
                "Respond with JSON:\n"
                "{\"dimensions\": [{\"name\": \"...\", \"score\": N, "
                "\"issues\": [...], \"suggestions\": [...]}], "
                "\"overall_score\": N, \"decision\": \"PASS or REVISE\", "
                "\"feedback_summary\": \"...\"}"},
            {"role": "user", "content":
                f"QUESTION: {question}\n\nREPORT:\n{report}\n\nSOURCES:\n{src_text}"},
        ]
        raw = generate(messages, max_new_tokens=600, temperature=0.3, do_sample=True)

        # Parse review output
        try:
            match = re.search(r'\{[\s\S]*\}', raw)
            if match:
                review = json.loads(match.group())
            else:
                review = self._default_review()
        except (json.JSONDecodeError, ValueError):
            review = self._default_review()

        # Ensure decision consistency with scores
        dims = review.get("dimensions", [])
        if dims:
            scores = [d.get("score", 5) for d in dims]
            avg = sum(scores) / len(scores)
            review["overall_score"] = round(avg, 1)
            if avg >= 7.0 and all(s >= 6 for s in scores):
                review["decision"] = "PASS"
            else:
                review["decision"] = "REVISE"

        if verbose:
            print(f"[Reviewer] Evaluation complete:")
            for d in review.get("dimensions", []):
                print(f"  {d.get('name', '?'):<20s} {d.get('score', '?')}/10")
            print(f"  Overall: {review.get('overall_score', '?')}/10 → {review.get('decision', '?')}")

        return review

    def format_feedback(self, review: dict) -> str:
        """Format review into feedback string for the writer."""
        lines = ["REVIEWER FEEDBACK:", ""]
        for d in review.get("dimensions", []):
            name = d.get("name", "unknown")
            score = d.get("score", "?")
            lines.append(f"{name}: {score}/10")
            for issue in d.get("issues", []):
                lines.append(f"  Issue: {issue}")
            for sug in d.get("suggestions", []):
                lines.append(f"  Fix: {sug}")
            lines.append("")
        lines.append(f"Decision: {review.get('decision', '?')}")
        lines.append(f"Summary: {review.get('feedback_summary', '')}")
        return "\n".join(lines)

    def _default_review(self) -> dict:
        return {
            "dimensions": [{"name": d, "score": 5, "issues": [], "suggestions": []}
                           for d in self.DIMENSIONS],
            "overall_score": 5.0,
            "decision": "REVISE",
            "feedback_summary": "Could not parse review; defaulting to REVISE.",
        }

reviewer = ReviewerAgent()
print("✓ ReviewerAgent ready")

## Part 4: The Orchestrator — Coordinating Everything

The Orchestrator is the conductor. It manages the workflow, tracks state, handles errors, and enforces the revision loop.

In [ ]:
@dataclass
class StageResult:
    """Result of one pipeline stage."""
    stage: str
    success: bool
    output: Any
    elapsed: float
    error: str = ""

class OrchestratorAgent:
    """Orchestrates the full Castalia Scholar pipeline."""

    STAGES = ["plan", "retrieve", "analyze", "write", "review"]

    def __init__(self, planner, retriever, analyzer, writer, reviewer,
                 max_revisions: int = 2):
        self.planner = planner
        self.retriever = retriever
        self.analyzer = analyzer
        self.writer = writer
        self.reviewer = reviewer
        self.max_revisions = max_revisions
        self.stage_results = []
        self.revision_count = 0

    def run(self, question: str, verbose: bool = True) -> dict:
        """Run the full research pipeline on a question."""
        total_start = time.time()
        self.stage_results = []
        self.revision_count = 0

        if verbose:
            print("╔" + "═" * 58 + "╗")
            print("║  CASTALIA SCHOLAR — Full System Execution" + " " * 16 + "║")
            print("╠" + "═" * 58 + "╣")
            print(f"║  Question: {question[:45]:<47s}║")
            print("╚" + "═" * 58 + "╝")
            print()

        # ── Stage 1: PLAN ──
        sub_questions = self._run_stage(
            "plan", lambda: self.planner.plan(question, verbose=verbose), verbose)
        if sub_questions is None:
            sub_questions = [question]

        # ── Stage 2: RETRIEVE ──
        passages = self._run_stage(
            "retrieve", lambda: self.retriever.retrieve(sub_questions, verbose=verbose),
            verbose)
        if passages is None:
            return self._build_result(question, None, total_start, "Retrieval failed")

        # ── Stage 3: ANALYZE ──
        analysis = self._run_stage(
            "analyze",
            lambda: self.analyzer.analyze(question, passages, verbose=verbose),
            verbose)
        if analysis is None:
            analysis = {"synthesis": "Analysis unavailable", "key_findings": [],
                        "themes": [], "contradictions": [], "gaps": []}

        # ── Stage 4: WRITE ──
        report = self._run_stage(
            "write",
            lambda: self.writer.write(question, analysis, passages, verbose=verbose),
            verbose)
        if report is None:
            return self._build_result(question, None, total_start, "Writing failed")

        # ── Stage 5: REVIEW + REVISION LOOP ──
        for rev_round in range(self.max_revisions + 1):
            review = self._run_stage(
                f"review_round_{rev_round}",
                lambda: self.reviewer.review(report, passages, question, verbose=verbose),
                verbose)
            if review is None:
                break  # Review failed, deliver report as-is

            decision = review.get("decision", "PASS")
            if decision == "PASS":
                if verbose:
                    print(f"\n  ✓ Report PASSED review at round {rev_round}")
                break

            if rev_round < self.max_revisions:
                self.revision_count += 1
                if verbose:
                    print(f"\n  ↻ Revision round {self.revision_count}/{self.max_revisions}")
                feedback = self.reviewer.format_feedback(review)
                report = self._run_stage(
                    f"revise_round_{rev_round}",
                    lambda: self.writer.revise(report, feedback, question, passages,
                                               verbose=verbose),
                    verbose)
                if report is None:
                    break
            else:
                if verbose:
                    print(f"\n  ⚠ Max revisions ({self.max_revisions}) reached, delivering best version")

        return self._build_result(question, report, total_start)

    def _run_stage(self, stage_name: str, fn: Callable, verbose: bool) -> Any:
        """Run a pipeline stage with timing and error handling."""
        if verbose:
            print(f"\n{'─' * 60}")
            print(f"  Stage: {stage_name.upper()}")
            print(f"{'─' * 60}")
        t0 = time.time()
        try:
            result = fn()
            elapsed = time.time() - t0
            self.stage_results.append(StageResult(stage_name, True, result, elapsed))
            if verbose:
                print(f"  ✓ {stage_name} completed in {elapsed:.1f}s")
            return result
        except Exception as e:
            elapsed = time.time() - t0
            self.stage_results.append(StageResult(stage_name, False, None, elapsed, str(e)))
            if verbose:
                print(f"  ✗ {stage_name} FAILED after {elapsed:.1f}s: {e}")
            return None

    def _build_result(self, question: str, report: Optional[str],
                      total_start: float, error: str = "") -> dict:
        """Build the final result dictionary."""
        total_time = time.time() - total_start
        return {
            "question": question,
            "report": report,
            "stages": self.stage_results,
            "total_time": total_time,
            "revision_rounds": self.revision_count,
            "error": error,
        }

    def print_metrics(self, result: dict):
        """Print execution metrics dashboard."""
        print(f"\n{'═' * 60}")
        print("EXECUTION METRICS")
        print(f"{'═' * 60}")
        print(f"Question: {result['question'][:50]}...")
        print(f"Total time: {result['total_time']:.1f}s")
        print(f"Revision rounds: {result['revision_rounds']}")
        print(f"Stages executed: {len(result['stages'])}")
        print()
        print(f"{'Stage':<25s} {'Status':<10s} {'Time':<10s}")
        print("-" * 45)
        for sr in result["stages"]:
            status = "✓" if sr.success else "✗"
            print(f"  {sr.stage:<23s} {status:<10s} {sr.elapsed:.1f}s")
        if result["error"]:
            print(f"\nError: {result['error']}")

orchestrator = OrchestratorAgent(planner, retriever, analyzer, writer, reviewer)
print("✓ OrchestratorAgent ready — all agents connected")

## Part 5: End-to-End Execution

### Research Question 1: Large Language Model Training

Let's run the complete system on a real research question and watch every agent in action.

In [ ]:
# ── Research Question 1 ──
result_1 = orchestrator.run(
    "What are the main approaches to training large language models?",
    verbose=True,
)

orchestrator.print_metrics(result_1)

In [ ]:
print("FINAL REPORT — Research Question 1")
print("=" * 60)
if result_1["report"]:
    print(result_1["report"])
else:
    print("No report was generated.")

### Research Question 2: Computer Vision Evolution

A different topic to test the system's generalization across domains.

In [ ]:
result_2 = orchestrator.run(
    "How has computer vision evolved from CNNs to vision transformers?",
    verbose=True,
)

orchestrator.print_metrics(result_2)

print(f"\n{'=' * 60}")
print("FINAL REPORT — Research Question 2")
print("=" * 60)
if result_2["report"]:
    print(result_2["report"][:1500] + "..." if len(result_2["report"]) > 1500 else result_2["report"])
else:
    print("No report generated.")

### Research Question 3: Reinforcement Learning Trade-offs

A question that requires comparing and contrasting — testing the analyzer's contradiction detection.

In [ ]:
result_3 = orchestrator.run(
    "What are the trade-offs between different approaches to reinforcement learning?",
    verbose=True,
)

orchestrator.print_metrics(result_3)

print(f"\n{'=' * 60}")
print("FINAL REPORT — Research Question 3")
print("=" * 60)
if result_3["report"]:
    print(result_3["report"][:1500] + "..." if len(result_3["report"]) > 1500 else result_3["report"])
else:
    print("No report generated.")

## Part 6: System Metrics Dashboard

Let's compare performance across all three research questions.

In [ ]:
# Build metrics dashboard
all_results = [
    ("LLM Training", result_1),
    ("CV Evolution", result_2),
    ("RL Trade-offs", result_3),
]

print("CASTALIA SCHOLAR — SYSTEM METRICS DASHBOARD")
print("=" * 70)
print(f"{'Question':<20s} {'Time':<10s} {'Stages':<10s} {'Revisions':<12s} {'Report Len':<12s}")
print("-" * 70)

for name, result in all_results:
    stages = len(result["stages"])
    revisions = result["revision_rounds"]
    report_len = len(result["report"]) if result["report"] else 0
    print(f"{name:<20s} {result['total_time']:<10.1f} {stages:<10d} {revisions:<12d} {report_len:<12d}")

print("-" * 70)

# Stage timing breakdown
print(f"\nSTAGE TIMING BREAKDOWN (seconds)")
print(f"{'Stage':<25s}", end="")
for name, _ in all_results:
    print(f" {name:<15s}", end="")
print()
print("-" * 70)

# Collect unique stage names across all runs
all_stage_names = []
for _, result in all_results:
    for sr in result["stages"]:
        if sr.stage not in all_stage_names:
            all_stage_names.append(sr.stage)

for stage_name in all_stage_names:
    print(f"  {stage_name:<23s}", end="")
    for _, result in all_results:
        found = False
        for sr in result["stages"]:
            if sr.stage == stage_name:
                print(f" {sr.elapsed:<15.1f}", end="")
                found = True
                break
        if not found:
            print(f" {'—':<15s}", end="")
    print()

In [ ]:
# Extract review scores from stage results
print("\nREVIEW SCORES ACROSS RUNS")
print("=" * 70)

for name, result in all_results:
    print(f"\n{name}:")
    for sr in result["stages"]:
        if sr.stage.startswith("review") and sr.success and sr.output:
            review = sr.output
            dims = review.get("dimensions", [])
            decision = review.get("decision", "?")
            overall = review.get("overall_score", "?")
            print(f"  {sr.stage}: overall={overall}/10, decision={decision}")
            for d in dims:
                bar = "█" * int(d.get("score", 0)) + "░" * (10 - int(d.get("score", 0)))
                print(f"    {d.get('name', '?'):<20s} [{bar}] {d.get('score', '?')}/10")

## Part 7: The Big Comparison — Single LLM vs. RAG vs. Full System

This is the payoff of the entire course. We answer the **same question** three different ways and compare the results.

| Approach | What It Does | Course Module |
|----------|-------------|---------------|
| **Single LLM call** | Ask the LLM directly (no tools, no retrieval) | Module 1: Prompt Engineering |
| **RAG retrieval + answer** | Retrieve relevant docs, include in prompt | Module 4: RAG |
| **Full Castalia Scholar** | Plan → Retrieve → Analyze → Write → Review | Module 7: Capstone |

In [ ]:
comparison_question = "What are the main approaches to training large language models?"

print("THREE-WAY COMPARISON")
print("=" * 70)

# ── Approach 1: Single LLM call ──
print("\n" + "─" * 60)
print("APPROACH 1: Single LLM Call (no retrieval, no agents)")
print("─" * 60)
t0 = time.time()
single_messages = [
    {"role": "system", "content": "You are a helpful AI assistant."},
    {"role": "user", "content": f"Write a research report on: {comparison_question}"},
]
single_report = generate(single_messages, max_new_tokens=600, temperature=0.7)
single_time = time.time() - t0
print(f"Time: {single_time:.1f}s")
print(f"Length: {len(single_report)} chars")
print(f"Preview: {single_report[:300]}...")

# ── Approach 2: Simple RAG ──
print("\n" + "─" * 60)
print("APPROACH 2: RAG (retrieve + answer in one call)")
print("─" * 60)
t0 = time.time()
# Retrieve relevant passages
query_emb = embed_texts([comparison_question]).astype(np.float32)
scores, indices = faiss_index.search(query_emb, k=5)
rag_context = "\n\n".join(
    f"[Source {i+1}: {CORPUS[idx]['title']}] {CORPUS[idx]['text']}"
    for i, idx in enumerate(indices[0])
)
rag_messages = [
    {"role": "system", "content": "Answer the question using the provided sources. Cite sources as [Source N]."},
    {"role": "user", "content": f"Question: {comparison_question}\n\nSources:\n{rag_context}"},
]
rag_report = generate(rag_messages, max_new_tokens=600, temperature=0.7)
rag_time = time.time() - t0
print(f"Time: {rag_time:.1f}s")
print(f"Length: {len(rag_report)} chars")
print(f"Preview: {rag_report[:300]}...")

# ── Approach 3: Full System (already ran as result_1) ──
print("\n" + "─" * 60)
print("APPROACH 3: Full Castalia Scholar (5 agents + review loop)")
print("─" * 60)
system_report = result_1["report"] or ""
system_time = result_1["total_time"]
print(f"Time: {system_time:.1f}s")
print(f"Length: {len(system_report)} chars")
print(f"Revisions: {result_1['revision_rounds']}")
print(f"Preview: {system_report[:300]}...")

In [ ]:
# Use LLM as judge to evaluate all three reports
def evaluate_report(report: str, question: str, label: str) -> dict:
    """Use LLM-as-judge to evaluate a report on key dimensions."""
    messages = [
        {"role": "system", "content":
            "You are evaluating a research report. Score these dimensions 1-10:\n"
            "- quality: overall writing quality\n"
            "- depth: how deeply topics are covered\n"
            "- citations: are claims attributed to sources (0 if none)\n"
            "- accuracy: factual correctness\n"
            "- completeness: coverage of the topic\n\n"
            "Respond ONLY with JSON: "
            "{\"quality\": N, \"depth\": N, \"citations\": N, "
            "\"accuracy\": N, \"completeness\": N}"},
        {"role": "user", "content": f"Question: {question}\n\nReport:\n{report[:1500]}"},
    ]
    raw = generate(messages, max_new_tokens=200, temperature=0.3, do_sample=True)
    try:
        match = re.search(r'\{[\s\S]*\}', raw)
        if match:
            scores = json.loads(match.group())
            return {k: min(10, max(1, int(v))) for k, v in scores.items()
                    if k in ("quality", "depth", "citations", "accuracy", "completeness")}
    except (json.JSONDecodeError, ValueError):
        pass
    return {"quality": 5, "depth": 5, "citations": 5, "accuracy": 5, "completeness": 5}

print("Evaluating all three approaches with LLM-as-judge...")
print()

eval_single = evaluate_report(single_report, comparison_question, "Single LLM")
print(f"Single LLM:     {eval_single}")

eval_rag = evaluate_report(rag_report, comparison_question, "RAG")
print(f"RAG:            {eval_rag}")

eval_system = evaluate_report(system_report, comparison_question, "Full System")
print(f"Full System:    {eval_system}")

In [ ]:
# Build the final comparison table
print("\n" + "=" * 75)
print("FINAL COMPARISON: Single LLM vs. RAG vs. Full Castalia Scholar")
print("=" * 75)

approaches = [
    ("Single LLM", single_report, single_time, eval_single, 1),
    ("RAG", rag_report, rag_time, eval_rag, 2),
    ("Full System", system_report, system_time, eval_system,
     len(result_1["stages"])),
]

# Header
print(f"\n{'Metric':<25s} {'Single LLM':<15s} {'RAG':<15s} {'Full System':<15s}")
print("-" * 70)

# Time
print(f"{'Time (seconds)':<25s}", end="")
for _, _, t, _, _ in approaches:
    print(f" {t:<14.1f}", end="")
print()

# Report length
print(f"{'Report length (chars)':<25s}", end="")
for _, r, _, _, _ in approaches:
    print(f" {len(r):<14d}", end="")
print()

# LLM calls
print(f"{'LLM calls':<25s}", end="")
for _, _, _, _, calls in approaches:
    print(f" {calls:<14d}", end="")
print()

# Quality dimensions
for dim in ["quality", "depth", "citations", "accuracy", "completeness"]:
    print(f"{dim.capitalize():<25s}", end="")
    for _, _, _, ev, _ in approaches:
        score = ev.get(dim, "?")
        bar = "█" * int(score) + "░" * (10 - int(score)) if isinstance(score, int) else "?"
        print(f" {score}/10 {bar:<4s}", end="")
    print()

# Average
print(f"\n{'AVERAGE':<25s}", end="")
for _, _, _, ev, _ in approaches:
    vals = [v for v in ev.values() if isinstance(v, (int, float))]
    avg = sum(vals) / len(vals) if vals else 0
    print(f" {avg:<14.1f}", end="")
print()

print("-" * 70)
print("\nThe full multi-agent system trades time for quality:")
print("  • More LLM calls → higher cost, but each agent is specialized")
print("  • Retrieval grounds the report in factual sources")
print("  • Review loop catches and corrects errors before delivery")
print("  • The result: deeper, more accurate, better-cited research reports")

## Part 8: Course Conclusion — What We've Built

### The Journey

Over 37 notebooks, we built a complete AI agent system from first principles:

| Module | Notebooks | What We Learned |
|--------|-----------|-----------------|
| **1. Foundations** | 01–03 | Prompt engineering, structured output, tool use |
| **2. Core Patterns** | 04–09 | ReAct, planning, reflection, multi-agent debate |
| **3. Memory & RAG** | 10–18 | Conversation memory, embeddings, vector search, chunking |
| **4. Advanced RAG** | 19–24 | Query decomposition, reranking, self-RAG, evaluation |
| **5. Agent Frameworks** | 25–28 | State machines, handoffs, supervision patterns |
| **6. Advanced Agents** | 29–31 | Streaming, human-in-the-loop, deployment |
| **7. Capstone** | 32–37 | Full multi-agent research system (Castalia Scholar) |

### What Castalia Scholar Demonstrates

1. **Decomposition works** — breaking a complex task into specialized agents produces better results than a single monolithic prompt
2. **Retrieval grounds generation** — connecting LLMs to source documents reduces hallucination
3. **Review improves quality** — the generate-critique-revise loop from Notebook 07 scales to system-level quality gates
4. **Orchestration enables complexity** — a well-designed orchestrator can manage multi-stage workflows with error handling and feedback loops

### Where to Go Next

- **Scale the corpus** — replace our 20-passage corpus with a real document database
- **Add web search** — integrate live search APIs for up-to-date information
- **Parallelize agents** — run retrieval and analysis concurrently for speed
- **Add human-in-the-loop** — let users guide the research direction between stages
- **Deploy as an API** — wrap the orchestrator in a FastAPI server
- **Evaluate systematically** — build evaluation benchmarks for each agent and the full system

## 🏋️ Exercises

Put your understanding to the test:

**Exercise 1 — Explore:** extend the agent with a new tool. Document what changes and why.

**Exercise 2 — Build:** add error handling for a failure mode discussed in this notebook. Compare your implementation to the one in this notebook.

**Exercise 3 — Extend:** trace through the agent loop manually and predict the output before running.

## Summary & Key Takeaways

### What We Built in This Notebook

| Component | Purpose |
|-----------|---------|
| **Document Corpus** | 20 AI/ML passages indexed with FAISS + BGE embeddings |
| **PlannerAgent** | Decomposes questions into sub-questions |
| **RetrieverAgent** | Semantic search with FAISS |
| **AnalyzerAgent** | Synthesis, contradiction detection, gap finding |
| **WriterAgent** | Report writing with citations + revision capability |
| **ReviewerAgent** | 5-dimension quality evaluation |
| **OrchestratorAgent** | Full pipeline coordination with error handling and revision loops |

### Key Results

1. The full system produces **longer, deeper, better-cited reports** than single LLM calls or simple RAG
2. The **review loop** catches errors and drives measurable quality improvement
3. **Specialized agents** outperform monolithic approaches because each can be prompted and optimized independently
4. The system is **robust** — error handling and fallbacks keep it running even when individual stages fail

### The Big Lesson

> **AI agents are software engineering applied to LLMs.** The same principles that make software systems reliable — decomposition, specialization, testing, error handling, iterative improvement — make agent systems reliable too.

---

*Congratulations on completing the Castalia Agents course! You've built a multi-agent research system from scratch, understanding every component from prompt engineering to system orchestration.*

## 📚 References & Further Reading

- [Hugging Face Documentation](https://huggingface.co/docs)
- [LLM Course Resources](https://github.com/cyruslayo/castalia)
- Explore related notebooks in the agents/ directory

---

## 🧭 Navigation

⬅️ **Previous:** [36 Project Review Agent](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/agents/36_project_review_agent.ipynb)